# ITO5202 Assessment 1 — Analysing Historical Data with System Performance

**Student ID:** 29701201  **Unit:** ITO5202
**Dataset:** Brazilian E-Commerce Public Dataset by Olist ([Kaggle](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce))

## Environment Setup 

Spark runs in local mode inside a Docker container, with the driver acting as the single executor. The configuration choices below are made deliberately because they affect every later measurement:

- `local[*]` uses every core the container exposes, so `defaultParallelism` equals the container's core count.
- `spark.driver.memory` must be set before the JVM starts, so this cell should be the first Spark call after a kernel restart.
- `spark.sql.session.timeZone = UTC`. The Olist timestamps are Brazilian local times with no zone attached. Parsing them in UTC stops the container's time zone (and any daylight-saving gaps) from silently shifting or nulling timestamps, which would otherwise change quarter boundaries and delivery delays.
- `spark.sql.shuffle.partitions` is reduced from the default 200 to 2 × cores. The full pipeline processes roughly 110k line items. 200 shuffle partitions would create mostly tiny tasks whose scheduling overhead exceeds their work. 

In [1]:
# ---------------------------------------------------------------
# Imports
# ---------------------------------------------------------------
import os          # file paths and CPU count
import platform    # Python / OS version for the environment table

import pandas as pd  

# Spark configuration and entry points
from pyspark import SparkConf
from pyspark.sql import SparkSession

# DataFrame functions, window specifications and schema types
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (StructType, StructField, StringType,
                               IntegerType, DoubleType, TimestampType)

In [2]:
# ---------------------------------------------------------------
# Spark configuration
# ---------------------------------------------------------------

master = "local[*]"


app_name = "ITO5202_A1_Olist_29701201"

# Set up configuration parameters for Spark
spark_conf = (
    SparkConf()
    .setMaster(master)
    .setAppName(app_name)
    # Driver memory: in local mode the driver is also the executor, so this
    # is all the memory Spark has. It only takes effect when the JVM starts,
    # so restart the kernel before running this cell.
    .set("spark.driver.memory", "4g")
    # Parse timestamps exactly as written in the CSVs (no time-zone shifting)
    .set("spark.sql.session.timeZone", "UTC")
    # Hide console progress bars so the PDF export stays clean
    .set("spark.ui.showConsoleProgress", "false")
)

# ---------------------------------------------------------------
# SparkSession 
# ---------------------------------------------------------------
spark = SparkSession.builder.config(conf=spark_conf).getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("ERROR")  # show errors only, to keep outputs readable

# ---------------------------------------------------------------
# Shuffle partitions
# ---------------------------------------------------------------
# After a shuffle (groupBy, join, window), data is split into this many
# partitions. 2 x cores gives each core about two tasks: enough to keep all
# cores busy without the overhead of the default 200 near-empty tasks.
SHUFFLE_PARTITIONS = sc.defaultParallelism * 2
spark.conf.set("spark.sql.shuffle.partitions", SHUFFLE_PARTITIONS)

print(f"Spark {spark.version} session started: {app_name}")
print(f"Shuffle partitions set to {SHUFFLE_PARTITIONS} "
      f"({sc.defaultParallelism} cores x 2)")

Spark 4.1.1 session started: ITO5202_A1_Olist_29701201
Shuffle partitions set to 16 (8 cores x 2)


In [3]:
# ---------------------------------------------------------------
# Execution environment summary 
# ---------------------------------------------------------------
def container_memory_limit():
    """Return the memory limit Docker has placed on this container, if any.
    Checks cgroup v2 first, then cgroup v1."""
    for path in ("/sys/fs/cgroup/memory.max",
                 "/sys/fs/cgroup/memory/memory.limit_in_bytes"):
        try:
            raw = open(path).read().strip()
            if raw != "max" and int(raw) < 1 << 60:   # very large value = no limit
                return f"{int(raw) / 1024**3:.1f} GB"
            return "no limit set"
        except (OSError, ValueError):
            continue
    return "unknown"


env = {
    "Execution mode": f"local ({sc.master}), Docker container",
    "Spark version": spark.version,
    "Python version": platform.python_version(),
    "OS (container)": f"{platform.system()} {platform.release()}",
    "CPU cores visible to container": os.cpu_count(),
    "defaultParallelism": sc.defaultParallelism,
    "Driver memory": spark.conf.get("spark.driver.memory", "default (1g)"),
    "Container memory limit": container_memory_limit(),
    "spark.sql.shuffle.partitions": spark.conf.get("spark.sql.shuffle.partitions"),
    # AQE can re-optimise plans at runtime (e.g. coalescing partitions,
    # switching join strategy). It appears as AdaptiveSparkPlan in explain().
    "spark.sql.adaptive.enabled (AQE)": spark.conf.get("spark.sql.adaptive.enabled"),
    # Tables smaller than this are broadcast to every task instead of shuffled
    "spark.sql.autoBroadcastJoinThreshold": spark.conf.get("spark.sql.autoBroadcastJoinThreshold"),
    "Session time zone": spark.conf.get("spark.sql.session.timeZone"),
    "Spark Web UI": "http://localhost:4040",
}

pd.DataFrame(env.items(), columns=["Setting", "Value"])

,Setting,Value
0,Execution mode,"local (local[*]), Docker container"
1,Spark version,4.1.1
2,Python version,3.13.12
3,OS (container),Linux 7.0.12-linuxkit
4,CPU cores visible to container,8
5,defaultParallelism,8
6,Driver memory,4g
7,Container memory limit,no limit set
8,spark.sql.shuffle.partitions,16
9,spark.sql.adaptive.enabled (AQE),true
